# Destination allocator of final demand — breadth share, state product, or WiNDC final-demand shares

The nesting step of the construction splits every final-demand flow of the OECD ICIO that
terminates in the United States across the 51 state regions. This notebook is the evidence
behind the allocator `nest_v31.py` uses for that split, $\Theta$ of equation (Theta) of the data
descriptor, and it produces the figures of the corresponding Technical Validation subsection.

An earlier version of the construction used the **breadth share** $\theta_s$, the unweighted
cross-sector mean of the production-share matrix $S$,

$$\theta_s=\frac{\tfrac1{|\mathcal J|}\sum_j S_{s,j}}{\sum_{s'}\tfrac1{|\mathcal J|}\sum_j S_{s',j}},
\qquad
F^n_{(c,i),(s,\text{cat})}=F^{O}_{(c,i),\mathrm{USA}_{\text{cat}}}\,\theta_s ,
\qquad
F^n_{(s,i),(s',\text{cat})}=F^{O}_{\mathrm{USA}_i,\mathrm{USA}_{\text{cat}}}\,S_{s,i}\,\theta_{s'} .$$

which understates large consuming states. The comparison below makes that quantitative and
settles the replacement; `nest_v31.F_SOURCE = "breadth"` reproduces the earlier behaviour.

**Three allocators are compared**, all renormalised to sum to one over the 51 states so that each
one is applied to the *same* OECD national totals and conserves them exactly:

| | allocator | granularity | source |
|---|---|---|---|
| **A** | breadth share $\theta_s$ (delivered series) | state | BEA SAGDP2, unweighted sector mean |
| **B** | state product share $\theta^{\mathrm{GDP}}_s$ | state | BEA SAGDP2, all-industry total |
| **C** | WiNDC final-demand share $\theta^{W}_{s,c}$ | state $\times$ category | WiNDC `cd0`/`i0`/`g0` |

A fourth, finer variant $\theta^{W}_{s,c,j}$ (state $\times$ category $\times$ sector) is also
evaluated in §8.

**External referent.** WiNDC cannot be scored against itself. The referents used are stated in
§1.3 and are, per category, the official BEA/Census regional statistic for the concept being
allocated. §5 tests the government referent against three independent alternatives, and §10
states plainly what this comparison can and cannot establish.

In [ ]:
%matplotlib inline
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "src"))   # `paths` without an install

import io, json, zipfile, urllib.request
import numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy import stats
import pyarrow.parquet as pq
import warnings; warnings.filterwarnings("ignore")

from paths import ROOT

YEAR          = 2017                       # reference year of the manuscript
YEARS         = list(range(1997, 2023))     # delivered series
WINDC_VERSION = "v3.1_RAS"
WINDC_SCALE   = 1000.0                      # WiNDC Bn$ -> OECD M$
FD_CATS       = ["DPABR", "GFCF", "GGFC", "HFCE", "INVNT", "NPISH"]
WCATS         = ["C", "I", "G"]             # WiNDC final-demand columns, in stored order

IOT_USA    = ROOT / "data/interim/IOT/IOT_USA"
WINDC_ST   = IOT_USA / f"grav_fric_{WINDC_VERSION}"   # all years, 71 sectors
# Only the final-demand block F is read from this build, and the column calibration
# of ras_table rewrites Z, tls_int and taxes but never F -- so the calibrated build
# carries the same F as the pre-RAS one, and the pre-RAS screening build (which the
# pipeline does not deliver) is not needed here.
WINDC_AGG  = IOT_USA / f"grav_fric_{WINDC_VERSION}_aggregated"
WINDC_HARM = IOT_USA / f"grav_fric_{WINDC_VERSION}_harmonized"
OECD_AGG   = ROOT / "data/interim/IOT/OCDE ICIO aggregated"
SAGDP2     = ROOT / "data/raw/BEA/SAGDP/SAGDP2__ALL_AREAS_1997_2025.csv"
SAPCE_DIR  = ROOT / "data/raw/BEA/SAPCE"
FIG_DIR    = ROOT / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

# house palette (same as source_comparison_oecd_windc.ipynb)
CB, CO_, CG, CP, CK = "#3577a8", "#bd5e2c", "#4c9a52", "#8a5aa8", "#444444"
COL = {"breadth": CO_, "gdp": CB, "windc": CG, "ref": CK}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})

RESULTS = {}     # every number quoted in the conclusion is stored here
print("ROOT:", ROOT)

---
## 1. Preliminary checks

Three things must hold before the substitution can even be considered: the WiNDC final demand
must exist at the required granularity, the WiNDC $\leftrightarrow$ OECD category correspondence
must be defined, and an external referent must be available.

### 1.1 Is the WiNDC final demand available at state $\times$ category?

Yes. The final-demand block of the intra-US table is stored as
`F[(origin state, origin sector), (destination state, category)]`, of shape
$(51\times J)\times(51\times 3)$, with the three WiNDC categories $C$ (`cd0`), $I$ (`i0`) and
$G$ (`g0`). Summing over origin rows gives the destination mass by state and category; keeping
the origin sector gives the finer state $\times$ category $\times$ sector variant.

The block is available for every year 1997–2022 at 71 WiNDC sectors, and for 2017 at the 36
aggregated sectors, harmonised and unharmonised. Because harmonisation (step H3) applies one
multiplier **per sector, uniform across states**, the state shares *within a sector* are identical
before and after harmonisation; only shares aggregated over sectors move, through the change in
sector composition. That residual move is measured below: it is negligible on the household
category and reaches a few tenths of a point on investment, which is the category with no external
referent. The harmonised shares are used for the reference year and the state-level files for the
time series.

In [ ]:
def windc_states_secs(npz):
    st = [str(x) for x in npz["regions"]]
    key = "proposed_sectors" if "proposed_sectors" in npz.files else "sectors"
    return st, [str(x) for x in npz[key]]

def windc_F_dest(npz, by_sector=False):
    """Destination mass of final demand, in Bn$.
    by_sector=False -> (state, cat);  True -> (origin sector, state, cat)."""
    st, sc = windc_states_secs(npz)
    F = npz["F"].reshape(len(st), len(sc), len(st), 3)
    return (F.sum(0) if by_sector else F.sum((0, 1))), st, sc

npz_h = np.load(WINDC_HARM / f"IOT_{YEAR}_harmonized.npz", allow_pickle=True)
npz_a = np.load(WINDC_AGG  / f"IOT_{YEAR}.npz",            allow_pickle=True)
npz_s = np.load(WINDC_ST   / f"IOT_{YEAR}.npz",            allow_pickle=True)

dest_h, STATES, SECS36 = windc_F_dest(npz_h)
dest_a, _, _           = windc_F_dest(npz_a)
dest_s, _, SECS71      = windc_F_dest(npz_s)

avail = pd.DataFrame(
    {"F block shape":     [str(np.load(WINDC_HARM / f'IOT_{YEAR}_harmonized.npz')['F'].shape),
                           str(npz_a['F'].shape), str(npz_s['F'].shape)],
     "sectors":           [len(SECS36), len(SECS36), len(SECS71)],
     "years on disk":     ["2017", "2017", f"{min(YEARS)}–2023"],
     "C total (bn $)":    [dest_h[:, 0].sum(), dest_a[:, 0].sum(), dest_s[:, 0].sum()],
     "I total (bn $)":    [dest_h[:, 1].sum(), dest_a[:, 1].sum(), dest_s[:, 1].sum()],
     "G total (bn $)":    [dest_h[:, 2].sum(), dest_a[:, 2].sum(), dest_s[:, 2].sum()]},
    index=["harmonised (36 sec)", "aggregated (36 sec)", "state level (71 sec)"])
print(f"51 states x 3 categories present for {YEAR}: "
      f"{dest_h.shape == (51, 3)}   states={len(STATES)}")
display(avail.round(1))

# state shares before / after harmonisation, aggregated over sectors
sh_h = dest_h / dest_h.sum(0); sh_a = dest_a / dest_a.sum(0)
mv = pd.DataFrame(np.abs(sh_h - sh_a) * 100, index=STATES, columns=WCATS)
print("\nShare move induced by harmonisation, pp — max over the 51 states, per category:")
print(mv.max().round(3).to_string())
print(f"  worst cell: {mv.stack().idxmax()} = {mv.values.max():.3f} pp"
      f"  |  NY, C: {sh_a[STATES.index('NY'),0]*100:.2f} -> "
      f"{sh_h[STATES.index('NY'),0]*100:.2f} pp")
RESULTS["harmonisation_max_share_move_pp"] = mv.max().round(4).to_dict()

### 1.2 Is the WiNDC $\leftrightarrow$ OECD category correspondence defined?

Partly, and the gap is a genuine limitation that must be stated. WiNDC carries **three**
final-demand categories, the OECD ICIO **six**. The grouping is unambiguous and is already
implemented in `nest_v31.py` (`windc_F_to_oecd`); the split *within* a group is not identified by
WiNDC and is taken from the OECD **national** sector profile, so it carries no state-specific
information.

In [ ]:
corr = pd.DataFrame([
    ("C  (cd0, household consumption)", "HFCE, NPISH, DPABR",
     "3 OECD categories in 1 WiNDC category", "split by OECD national sector profile"),
    ("I  (i0, investment)", "GFCF, INVNT",
     "2 OECD categories in 1 WiNDC category", "split by OECD national sector profile"),
    ("G  (g0, government)", "GGFC", "one to one", "exact"),
], columns=["WiNDC category", "OECD ICIO categories", "Mapping", "Within-group split"])
display(corr)

# how much OECD mass is affected by a non-identified within-group split
def oecd_usa_sectors(f):
    return [c.split("_", 1)[1] for c in pq.read_schema(f).names
            if c.startswith("USA_") and c.split("_", 1)[1] not in FD_CATS]

def pick_oecd_file(year, wd_secs):
    cands = list(OECD_AGG.rglob(f"{year}_*.parquet"))
    if not cands: raise FileNotFoundError(year)
    return max(cands, key=lambda f: len(set(oecd_usa_sectors(f)) & set(wd_secs)))

def load_oecd_fd(year, sect_order):
    """OECD USA final-demand blocks, M$: domestic (USA rows x USA FD) and imported
    (world rows x USA FD), on the given sector order."""
    f      = pick_oecd_file(year, sect_order)
    schema = pq.read_schema(f)
    usa_fd = [c for c in schema.names
              if c.startswith("USA_") and c.split("_", 1)[1] in FD_CATS]
    idx    = [c for c in (schema.pandas_metadata or {}).get("index_columns", [])
              if isinstance(c, str)]
    dfo    = pq.read_table(f, columns=usa_fd + idx).to_pandas()[usa_fd]
    cats   = [c.split("_", 1)[1] for c in usa_fd]
    urows  = [f"USA_{s}" for s in sect_order]
    wrows  = [r for r in dfo.index if isinstance(r, str) and "_" in r
              and len(r.split("_")[0]) == 3 and not r.startswith("USA_")]
    dom = pd.DataFrame(dfo.loc[urows].values, index=sect_order, columns=cats)
    imp = pd.Series(dfo.loc[wrows].values.sum(0), index=cats)
    return dom, imp, f.relative_to(ROOT)

FD_dom, FD_imp, oecd_file = load_oecd_fd(YEAR, SECS36)
print("OECD file:", oecd_file)
grp = {"C": ["HFCE", "NPISH", "DPABR"], "I": ["GFCF", "INVNT"], "G": ["GGFC"]}
tot = FD_dom.values.sum() + FD_imp.sum()
amb = sum(FD_dom[g].values.sum() + FD_imp[g].sum() for g in ["NPISH", "DPABR", "INVNT"])
print(f"\nOECD USA final demand {YEAR}: domestic {FD_dom.values.sum()/1e6:,.2f} tn$, "
      f"imported {FD_imp.sum()/1e6:,.2f} tn$")
print(f"Mass whose within-group split is NOT identified by WiNDC "
      f"(NPISH + DPABR + INVNT): {amb/tot*100:.1f}% of US final demand")
RESULTS["unidentified_within_group_pct"] = float(amb / tot * 100)
RESULTS["fd_domestic_M"] = float(FD_dom.values.sum())
RESULTS["fd_imported_M"] = float(FD_imp.sum())

### 1.3 External referent

WiNDC is one of the three candidates, so scoring against WiNDC would be circular. The referent
must therefore come from outside the construction. The WiNDC paper states how each final-demand
category is regionalised, and that dictates the choice:

| WiNDC category | regionalised on | external referent used here | status |
|---|---|---|---|
| $C$ | BEA **Personal Consumption Expenditures by state** | BEA regional table **SAPCE1**, line 1 (total PCE), 1997–2024 | direct measure of the concept |
| $G$ | Census **State Government Finances** | BEA **SAGDP2** line 83, government and government enterprises value added | *proxy* for the location of government activity |
| $I$ | BEA **gross state product** | none | **not identified** — no state-level investment statistic exists |

Two consequences must be carried into the conclusion. First, the $C$ referent is *not independent*
of candidate C: WiNDC's household shares are built from PCE, so close agreement is expected by
construction. The test therefore does not establish that WiNDC estimates consumption well — it
establishes which allocator **transmits** the official regional statistic and which one destroys it.
That is the question actually at issue. Second, the $I$ category (about 17% of domestic final demand)
carries no external check, and the government referent is a value-added proxy, not an expenditure
measure; both are stated as limits in §7.

In [ ]:
SAPCE_URL  = "https://apps.bea.gov/regional/zip/SAPCE.zip"
SAPCE_FILE = SAPCE_DIR / "SAPCE1__ALL_AREAS_1997_2024.csv"
if not SAPCE_FILE.exists():                      # one-off fetch, then cached in data/raw
    SAPCE_DIR.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(SAPCE_URL, timeout=120) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    for n in z.namelist():
        if n.startswith("SAPCE1__"): z.extract(n, SAPCE_DIR)
print("SAPCE1:", SAPCE_FILE.relative_to(ROOT), "|", f"{SAPCE_FILE.stat().st_size/1e3:.0f} kB")

STATE_NAME_TO_ABBR = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Connecticut':'CT','Delaware':'DE','District of Columbia':'DC',
    'Florida':'FL','Georgia':'GA','Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN',
    'Iowa':'IA','Kansas':'KS','Kentucky':'KY','Louisiana':'LA','Maine':'ME','Maryland':'MD',
    'Massachusetts':'MA','Michigan':'MI','Minnesota':'MN','Mississippi':'MS','Missouri':'MO',
    'Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH','New Jersey':'NJ',
    'New Mexico':'NM','New York':'NY','North Carolina':'NC','North Dakota':'ND','Ohio':'OH',
    'Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA','Rhode Island':'RI',
    'South Carolina':'SC','South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT',
    'Vermont':'VT','Virginia':'VA','Washington':'WA','West Virginia':'WV','Wisconsin':'WI',
    'Wyoming':'WY'}

def _bea_long(path, linecodes):
    df = pd.read_csv(path, dtype={"GeoFIPS": str})
    df["GeoName"] = df["GeoName"].astype(str).str.strip()
    df["abbr"] = df["GeoName"].map(STATE_NAME_TO_ABBR)
    df = df.dropna(subset=["abbr"])
    df = df[df["LineCode"].isin(linecodes)]
    ycols = [c for c in df.columns if c.isdigit()]
    df[ycols] = df[ycols].apply(pd.to_numeric, errors="coerce")
    return df, ycols

def pce_share(year):
    df, _ = _bea_long(SAPCE_FILE, [1])
    v = df.set_index("abbr")[str(year)].reindex(STATES)
    return v / v.sum()

def govva_share(year):
    df, _ = _bea_long(SAGDP2, [83])
    v = df.set_index("abbr")[str(year)].reindex(STATES)
    return v / v.sum()

ref_C, ref_G = pce_share(YEAR), govva_share(YEAR)
print(f"\nReferent C (PCE)  — NY {ref_C['NY']*100:.2f}%  CA {ref_C['CA']*100:.2f}%  "
      f"TX {ref_C['TX']*100:.2f}%   sum {ref_C.sum():.4f}")
print(f"Referent G (govVA) — NY {ref_G['NY']*100:.2f}%  CA {ref_G['CA']*100:.2f}%  "
      f"TX {ref_G['TX']*100:.2f}%   sum {ref_G.sum():.4f}")

---
## 2. The three allocators on a common basis

All three are vectors on the 51 states summing to one, so each is applied to the *same* OECD
national final-demand totals. Conservation of the OECD aggregate is therefore exact for all three
and cannot discriminate between them — only the distribution across states can.

In [ ]:
# SAGDP2 line-code -> common sector, as used by the delivered construction (nest_v31.py)
PROPOSED_TO_SAGDP2 = {
    "Accommodation and food service activities": [80, 81],
    "Administrative and support service activities": [66],
    "Agriculture, hunting, forestry, fishing and related": [4, 5],
    "Air transport": [37],
    "Arts, entertainment and recreation activities": [77, 78],
    "Construction": [11],
    "Education and public administration": [69, 84, 85, 86],
    "Electricity, gas, steam and air conditioning supply, and wasterwater": [10, 67],
    "Financial and insurance activities": [52, 53, 54, 55],
    "Human health and social work activities": [71, 72, 73, 74],
    "Land transport and transport via pipelines": [38, 40, 41, 42],
    "Manufacture of chemicals and chemical products, pharmaceutical": [32],
    "Manufacture of coke and refined petroleum products": [31],
    "Manufacture of computer, electronic and optical products": [19],
    "Manufacture of electrical equipment": [20],
    "Manufacture of fabricated metal products": [17],
    "Manufacture of food and beverage and tobacco products": [26],
    "Manufacture of furniture": [23],
    "Manufacture of machinery and equipment n.e.c. ": [18, 24],
    "Manufacture of motor vehicles, trailers and semi-trailers": [21],
    "Manufacture of other non-metallic mineral products": [15],
    "Manufacture of other transport equipment": [22],
    "Manufacture of paper and paper products": [29, 30],
    "Manufacture of rubber and plastic products": [33],
    "Manufacture of textiles, wearing apparel, leather and related products": [27, 28],
    "Manufacture of wood and of products of wood and cork": [14],
    "Mining, except oil & gas": [8],
    "Oil and gas extraction": [7],
    "Other services + wholesale/retail": [58, 82, 34, 35],
    "Professional, scientific and technical activities": [61, 63, 64],
    "Real estate activities": [57],
    "Support activities for mining": [9],
    "Warehousing and support activities for transportation": [43, 44],
    "Water transport": [39],
    "broadcasting, telecommunications, data processing, publishing, information services, "
    "motion picture, video, television": [46, 47, 48, 49, 62],
    "primary metals": [16],
}

_sag = pd.read_csv(SAGDP2, dtype={"GeoFIPS": str})
_sag["GeoName"] = _sag["GeoName"].str.strip()
_sag["abbr"] = _sag["GeoName"].map(STATE_NAME_TO_ABBR)
_sag = _sag.dropna(subset=["abbr"])
_YC = [c for c in _sag.columns if c.isdigit()]
_sag[_YC] = _sag[_YC].apply(pd.to_numeric, errors="coerce")
_L2S = {lc: s for s, ls in PROPOSED_TO_SAGDP2.items() for lc in ls}

def S_matrix(year):
    """Production-share matrix S[state, sector], columns summing to one (manuscript eq. 7)."""
    d = _sag[_sag.LineCode.isin(_L2S)].copy()
    d["sec"] = d.LineCode.map(_L2S)
    w = (d.groupby(["abbr", "sec"])[str(year)].sum(min_count=1)
           .unstack("sec").reindex(STATES).reindex(columns=SECS36))
    return w.div(w.sum(0), axis=1).fillna(0.0)

def breadth_share(year):
    """A — theta_s, unweighted cross-sector mean of S (manuscript eq. 8)."""
    t = S_matrix(year).mean(1)
    return t / t.sum()

def gdp_share(year):
    """B — state share of all-industry gross state product (SAGDP2 line 1)."""
    v = _sag[_sag.LineCode == 1].set_index("abbr")[str(year)].reindex(STATES)
    return v / v.sum()

def windc_share(year, harmonised=True, by_sector=False):
    """C — WiNDC final-demand destination shares, state x category (x sector)."""
    if harmonised and year == YEAR:
        npz = np.load(WINDC_HARM / f"IOT_{year}_harmonized.npz", allow_pickle=True)
    else:
        npz = np.load(WINDC_ST / f"IOT_{year}.npz", allow_pickle=True)
    d, st, sc = windc_F_dest(npz, by_sector=by_sector)
    if by_sector:                                  # (sector, state, cat)
        s = d / np.where(d.sum(1, keepdims=True) > 0, d.sum(1, keepdims=True), 1)
        return s, sc
    return pd.DataFrame(d / d.sum(0), index=st, columns=WCATS)

theta   = breadth_share(YEAR)
gdp_sh  = gdp_share(YEAR)
windc   = windc_share(YEAR)

ALLOC = {"breadth": pd.DataFrame({c: theta  for c in WCATS}),
         "gdp":     pd.DataFrame({c: gdp_sh for c in WCATS}),
         "windc":   windc}
LABEL = {"breadth": r"A  breadth share $\theta$", "gdp": r"B  state product share",
         "windc":   r"C  WiNDC final demand"}

chk = pd.DataFrame({k: v.sum() for k, v in ALLOC.items()})
print("column sums (must be 1 everywhere):"); display(chk.round(12))
display(pd.DataFrame({"A breadth": theta, "B gdp": gdp_sh,
                      "C WiNDC C": windc["C"], "C WiNDC I": windc["I"],
                      "C WiNDC G": windc["G"]}).mul(100)
        .loc[["CA", "TX", "NY", "FL", "IL", "PA", "OH", "WY", "VT", "DC"]].round(2))

---
## 3. Which final-demand blocks the allocator governs

A recurring ambiguity in the manuscript is worth settling before any comparison: the
allocator does **not** act on imported final demand alone. Equation (12) of the manuscript
uses it on the destination index of two blocks, and the larger of the two by far is the
*intra-United-States* one:

$$F^n_{(c,i),(s,\text{cat})}=F^{O}_{(c,i),\mathrm{USA}_{\text{cat}}}\,\theta_{s}
\quad\text{(imported)},\qquad
F^n_{(s,i),(s',\text{cat})}=F^{O}_{\mathrm{USA}_i,\mathrm{USA}_{\text{cat}}}\,S_{s,i}\,\theta_{s'}
\quad\text{(intra-US)}.$$

Two further points follow from the same equations and are easy to miss. The export block
$F^n_{(s,i),(c,\text{cat})}$ is split by the production share $S$ on the *origin* index only
and never sees the allocator. And the intra-United-States final-demand block of the delivered
series is **not** taken from the sub-national accounts at all: it is the OECD United States
domestic final demand redistributed by $S$ on the row and by the allocator on the column. The
cell below measures each block.

In [ ]:
def fd_block_masses(year, sect_order):
    """2017 masses of the four final-demand blocks plus the accounting rows, M$."""
    f    = pick_oecd_file(year, sect_order)
    sch  = pq.read_schema(f)
    idx  = [c for c in (sch.pandas_metadata or {}).get("index_columns", [])
            if isinstance(c, str)]
    ufd  = [c for c in sch.names if c.startswith("USA_") and c.split("_", 1)[1] in FD_CATS]
    wfd  = [c for c in sch.names if "_" in c and not c.startswith("USA_")
            and len(c.split("_")[0]) == 3 and c.split("_", 1)[1] in FD_CATS]
    d    = pq.read_table(f, columns=ufd + wfd + idx).to_pandas()
    isreg = lambda r: isinstance(r, str) and "_" in r and len(r.split("_")[0]) == 3
    ur = [r for r in d.index if isreg(r) and r.startswith("USA_")
          and r.split("_", 1)[1] not in FD_CATS]
    wr = [r for r in d.index if isreg(r) and not r.startswith("USA_")
          and r.split("_", 1)[1] not in FD_CATS]
    er = [r for r in d.index if r in ("TLS", "VA")]
    return pd.DataFrame([
        ("world → world", "foreign rows × foreign FD columns",
         "unchanged, copied from the OECD table", "—",
         d.loc[wr, wfd].values.sum()),
        ("world → states  (imported final demand)", "foreign rows × state FD columns",
         "destination state set by the allocator", "allocator",
         d.loc[wr, ufd].values.sum()),
        ("states → world  (exports for final use)", "state rows × foreign FD columns",
         "origin state set by the production share S", "S only",
         d.loc[ur, wfd].values.sum()),
        ("states → states (intra-US final demand)", "state rows × state FD columns",
         "origin by S, destination state by the allocator", "allocator",
         d.loc[ur, ufd].values.sum()),
        ("TLS and VA rows under state FD columns", "accounting rows × state FD columns",
         "split by the allocator", "allocator",
         d.loc[er, ufd].values.sum()),
    ], columns=["block", "index", "treatment in the delivered series", "role", "M$"])

BLK = fd_block_masses(YEAR, SECS36)
BLK["tn$"] = BLK["M$"] / 1e6
gov = BLK[BLK.role == "allocator"]["M$"].sum()
display(BLK[["block", "treatment in the delivered series", "tn$"]]
        .style.format({"tn$": "{:,.2f}"}).hide(axis="index"))

print(f"Mass whose DESTINATION state is fixed by the allocator: {gov/1e6:,.2f} tn$")
for lab, i in [("imported final demand", 1), ("intra-US final demand", 3),
               ("accounting rows", 4)]:
    print(f"   {lab:<24} {BLK.loc[i,'tn$']:6.2f} tn$   ({BLK.loc[i,'M$']/gov*100:4.1f}% of it)")
print(f"Allocated by S on the origin only (exports)   : {BLK.loc[2,'tn$']:.2f} tn$")
print(f"Never touched (world × world)                 : {BLK.loc[0,'tn$']:.2f} tn$")
print("\nThe intra-US block is 12x the imported block: the allocator is overwhelmingly a "
      "statement about where Americans consume, not about where imports land.")
RESULTS["fd_blocks_M"] = {r.block: float(r["M$"]) for _, r in BLK.iterrows()}
RESULTS["mass_governed_by_allocator_M"] = float(gov)

---
## 4. Characterising the bias

Visual inspection is not a criterion. The bias is defined on the **share error** against the
referent, $e_s=\hat\theta_s-\theta^{\text{ref}}_s$, which sums to zero by construction: any
under-allocation to one state is exactly an over-allocation elsewhere. Five statistics are
reported, each answering a different question.

| statistic | definition | reads as |
|---|---|---|
| $\mathrm{TV}$ | $\tfrac12\sum_s\lvert e_s\rvert$ | share of the national total sitting in the wrong state |
| $\mathrm{MIS}$ | $\mathrm{TV}\times$ national total | the same, in million dollars |
| $\beta$ | OLS slope of $e_s$ on $\theta^{\text{ref}}_s$ | **size bias**: $\beta<0$ means large states are systematically under-allocated |
| $\varepsilon$ | OLS slope of $\log\hat\theta_s$ on $\log\theta^{\text{ref}}_s$ | **compression elasticity**: $\varepsilon<1$ means the allocator flattens the distribution |
| $\rho,\ \rho_S$ | Pearson, Spearman | agreement on levels and on ranks |

$\beta$ is the statistic that formalises the claim under test. A mean-preserving allocator that is
merely noisy has $\beta\approx0$; an allocator that shrinks every state towards the uniform
$1/51$ has $\beta<0$ and $\varepsilon<1$.

In [ ]:
def metrics(m, ref, total=None):
    m, ref = np.asarray(m, float), np.asarray(ref, float)
    e = m - ref
    ok = (m > 0) & (ref > 0)
    out = {
        "TV":    float(np.abs(e).sum() / 2),
        "beta":  float(np.polyfit(ref, e, 1)[0]),
        "eps":   float(np.polyfit(np.log(ref[ok]), np.log(m[ok]), 1)[0]),
        "rho":   float(np.corrcoef(m, ref)[0, 1]),
        "rho_S": float(stats.spearmanr(m, ref).statistic),
        "top5":  float(np.sort(m)[::-1][:5].sum()),
        "HHI":   float((m ** 2).sum()),
    }
    if total is not None: out["MIS_M"] = out["TV"] * total
    return out

# category masses (OECD, M$) — domestic + imported, on the WiNDC 3-category grouping
MASS = {c: float(sum(FD_dom[g].values.sum() + FD_imp[g] for g in grp[c])) for c in WCATS}
print("OECD USA final demand by WiNDC category, M$:",
      {k: f"{v/1e6:,.2f} tn" for k, v in MASS.items()},
      f"| shares { {k: round(v/sum(MASS.values())*100,1) for k,v in MASS.items()} }")
RESULTS["mass_by_cat_M"] = MASS

REF = {"C": ref_C, "G": ref_G}
rows = []
for cat, ref in REF.items():
    for k, A in ALLOC.items():
        rows.append(dict(category=cat, allocator=k, **metrics(A[cat], ref, MASS[cat])))
    rows.append(dict(category=cat, allocator="referent", **metrics(ref, ref, MASS[cat])))
MET = pd.DataFrame(rows).set_index(["category", "allocator"])
display(MET[["TV", "MIS_M", "beta", "eps", "rho", "rho_S", "top5"]]
        .style.format({"TV": "{:.4f}", "MIS_M": "{:,.0f}", "beta": "{:+.4f}",
                       "eps": "{:.3f}", "rho": "{:.4f}", "rho_S": "{:.4f}",
                       "top5": "{:.3f}"}))
RESULTS["metrics_2017"] = MET.reset_index().to_dict("records")

### 4.1 New York and the other large consuming states

The claim in the Usage Notes is checked state by state on the household category, whose referent
is a direct measurement.

In [ ]:
BIG = ["CA", "TX", "NY", "FL", "PA", "IL", "OH", "NJ", "MI", "GA"]
tab = pd.DataFrame({"referent (PCE)": ref_C,
                    "A breadth": ALLOC["breadth"]["C"],
                    "B gdp":     ALLOC["gdp"]["C"],
                    "C WiNDC":   ALLOC["windc"]["C"]}).loc[BIG] * 100
for k in ["A breadth", "B gdp", "C WiNDC"]:
    tab[f"{k} err pp"] = tab[k] - tab["referent (PCE)"]
    tab[f"{k} err %"]  = (tab[k] / tab["referent (PCE)"] - 1) * 100
    tab[f"{k} err bn$"] = (tab[f"{k} err pp"] / 100) * MASS["C"] / 1e3
display(tab.round(2))

ny = tab.loc["NY"]
print(f"\nNEW YORK, household final demand {YEAR}:")
for k in ["A breadth", "B gdp", "C WiNDC"]:
    print(f"  {k:<10} {ny[k]:5.2f}%  vs referent {ny['referent (PCE)']:.2f}%  "
          f"-> {ny[k+' err pp']:+.2f} pp  ({ny[k+' err %']:+.1f}%,  "
          f"{ny[k+' err bn$']:+,.0f} bn$)")
RESULTS["NY_C_2017"] = {k: float(ny[k]) for k in tab.columns}

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))

# (a) share error against referent share — the size-bias regression
for k in ALLOC:
    e = (ALLOC[k]["C"] - ref_C) * 100
    ax[0].scatter(ref_C * 100, e, s=22, color=COL[k], alpha=.75, label=LABEL[k], zorder=3)
    b, a0 = np.polyfit(ref_C * 100, e, 1)
    xs = np.linspace(0, ref_C.max() * 100, 20)
    ax[0].plot(xs, a0 + b * xs, color=COL[k], lw=1.6, ls="--", alpha=.9)
for s in ["NY", "CA", "TX", "FL"]:
    ax[0].annotate(s, (ref_C[s] * 100, (ALLOC["breadth"]["C"][s] - ref_C[s]) * 100),
                   textcoords="offset points", xytext=(4, 4), fontsize=8, color=CO_)
ax[0].axhline(0, color="k", lw=.8)
ax[0].set(xlabel="referent share (BEA PCE), %", ylabel="share error, pp",
          title="a  size bias: error against state size")
ax[0].legend(fontsize=8, frameon=False)

# (b) allocated / measured, against state size — compression read multiplicatively
ax[1].axhline(1, color="k", lw=.9, ls=":", zorder=1)
xg = np.logspace(np.log10(ref_C.min() * 100), np.log10(ref_C.max() * 100), 30)
for k in ALLOC:
    r = ALLOC[k]["C"] / ref_C
    ax[1].scatter(ref_C * 100, r, s=22, color=COL[k], alpha=.75, zorder=3,
                  label=f"{LABEL[k]}  ε={metrics(ALLOC[k]['C'], ref_C)['eps']:.3f}")
    b, a0 = np.polyfit(np.log(ref_C * 100), np.log(r), 1)
    ax[1].plot(xg, np.exp(a0 + b * np.log(xg)), color=COL[k], lw=1.6, ls="--", alpha=.9)
for st in ["NY", "CA", "TX", "FL"]:
    ax[1].annotate(st, (ref_C[st] * 100, ALLOC["breadth"]["C"][st] / ref_C[st]),
                   textcoords="offset points", xytext=(4, 4), fontsize=8, color=CO_)
ax[1].set(xscale="log", xlabel="referent share (BEA PCE), %",
          ylabel="allocated / measured", title="b  compression, read multiplicatively")
ax[1].legend(fontsize=8, frameon=False, loc="lower left")

# (c) TV per category and allocator, in bn$
w, xs = .26, np.arange(len(REF))
for i, k in enumerate(ALLOC):
    v = [MET.loc[(c, k), "MIS_M"] / 1e3 for c in REF]
    bars = ax[2].bar(xs + (i - 1) * w, v, w, color=COL[k], label=LABEL[k])
    ax[2].bar_label(bars, fmt="%.0f", fontsize=8, padding=2)
ax[2].set(xticks=xs, xticklabels=[f"{c}  ({MASS[c]/1e6:,.1f} tn$)" for c in REF],
          ylabel="misallocated mass, bn $", title="c  mass in the wrong state")
ax[2].legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(FIG_DIR / "fd_allocator_bias.png", dpi=160,
                                bbox_inches="tight")
plt.show()

---
## 5. How good is the government referent?

The household referent measures its concept directly. The government referent does not: table
SAGDP2 reports government *value added*, which is compensation plus consumption of fixed
capital, whereas the category being allocated is government *final consumption expenditure*,
which adds intermediate purchases and covers federal as well as state and local government.
Since the government result is what decides between candidates C and D, it must not rest on a
single proxy. Four referents with different concept biases are therefore compared.

| | referent | source | what it measures | known bias |
|---|---|---|---|---|
| **R1** | government value added | BEA SAGDP2 line 83 | compensation + fixed-capital consumption, all levels | omits intermediate purchases, which are bought disproportionately by the federal government |
| **R2** | state and local current operations | Census of Governments 2017, table 1 | direct current spending of state and local government | excludes federal; includes Medicaid vendor payments, which inflate high-transfer states |
| **R3** | federal value added and Census current operations, weighted 36.6 / 63.4 | BEA SAGDP2 lines 84–85 and Census | closest available match to the concept, at the national federal / state-local split of the NIPA | mixes two sources; the federal part is still a value-added proxy |
| **R4** | direct general expenditure | Census of Governments 2017, table 1 | all direct state and local spending, capital included | broadest, furthest from a consumption concept |

R2 to R4 come from the Census of Governments, which is independent of the BEA. R1 is the only
one of the four available annually over the whole delivered series, so it is the one used for
the time series in §7; all four are used here on the reference year.

In [ ]:
CENSUS_DIR = ROOT / "data/raw/Census_gov_finances"
CENSUS_URL = ("https://www2.census.gov/programs-surveys/gov-finances/tables/2017/"
              "summary-tables/17slsstab{}.xlsx")
for part in ("1a", "1b"):                       # one-off fetch, then cached in data/raw
    p = CENSUS_DIR / f"17slsstab{part}.xlsx"
    if not p.exists():
        CENSUS_DIR.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(CENSUS_URL.format(part), timeout=120) as r:
            p.write_bytes(r.read())

def census_gov(rows={"current_ops": 80, "direct_gen": 90}):
    """State and local government finance by state, 2017, in thousands of dollars.
    Row numbers are those of Census table 1; the state columns are the 'State & local' ones."""
    out = []
    for part in ("1a", "1b"):
        d  = pd.read_excel(CENSUS_DIR / f"17slsstab{part}.xlsx", sheet_name=0, header=None)
        nm, lv = d.iloc[7, :].ffill(), d.iloc[8, :].astype(str)
        cols = {str(nm[j]).strip(): j for j in range(2, d.shape[1])
                if lv[j].strip().startswith("State & local")
                and str(nm[j]).strip() not in ("nan", "United States Total")}
        out.append(pd.DataFrame({k: pd.Series(
            {n: pd.to_numeric(d.iloc[i, j], errors="coerce") for n, j in cols.items()})
            for k, i in rows.items()}))
    C = pd.concat(out); C = C[~C.index.duplicated()]
    C.index = [STATE_NAME_TO_ABBR[i] for i in C.index]
    return C.reindex(STATES)

CEN = census_gov()
print(f"Census 2017, US totals: current operations "
      f"{CEN['current_ops'].sum()/1e6:,.0f} bn$, direct general expenditure "
      f"{CEN['direct_gen'].sum()/1e6:,.0f} bn$   ({len(CEN)} states)")

fed_va = _sag[_sag.LineCode.isin([84, 85])].groupby("abbr")[str(YEAR)].sum().reindex(STATES)
W_FED  = 0.366        # NIPA 2017: federal share of government consumption expenditures
nrm    = lambda x: x / x.sum()

GREFS = {
    "R1 government value added":            nrm(govva_share(YEAR)),
    "R2 Census S&L current operations":     nrm(CEN["current_ops"]),
    "R3 NIPA-weighted fed VA + Census S&L": W_FED * nrm(fed_va)
                                            + (1 - W_FED) * nrm(CEN["current_ops"]),
    "R4 Census direct general expenditure": nrm(CEN["direct_gen"]),
}
rows = []
for name, r in GREFS.items():
    row = {"referent": name, "NY": r["NY"] * 100, "CA": r["CA"] * 100, "TX": r["TX"] * 100}
    for k in ["breadth", "gdp", "windc"]:
        row[f"TV {k}"] = metrics(ALLOC[k]["G"], r)["TV"] * 100
    row["best"] = min(["breadth", "gdp", "windc"],
                      key=lambda k: metrics(ALLOC[k]["G"], r)["TV"])
    rows.append(row)
GR = pd.DataFrame(rows).set_index("referent")
display(GR.round(2))

print(f"\nAllocator shares for comparison:  NY  CA  TX")
for k, lab in [("windc", "C WiNDC G   "), ("gdp", "B product   "), ("breadth", "A breadth   ")]:
    v = ALLOC[k]["G"]
    print(f"  {lab} {v['NY']*100:5.2f} {v['CA']*100:5.2f} {v['TX']*100:5.2f}")
print(f"\nNew York's referent share spans {GR['NY'].min():.2f}-{GR['NY'].max():.2f}% across the "
      f"four referents; WiNDC allocates it {ALLOC['windc']['G']['NY']*100:.2f}%.")
print(f"The product share is the best of the three under {(GR['best']=='gdp').sum()} of 4 "
      f"referents and WiNDC the worst under {(GR[[c for c in GR if c.startswith('TV')]].idxmax(1)=='TV windc').sum()} of 4.")
RESULTS["gov_referents"] = GR.round(4).to_dict("index")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))

# (a) TV of each allocator under each government referent
w, xs = .26, np.arange(len(GREFS))
for i, k in enumerate(["breadth", "gdp", "windc"]):
    v = [GR.loc[n, f"TV {k}"] for n in GREFS]
    b = ax[0].bar(xs + (i - 1) * w, v, w, color=COL[k], label=LABEL[k])
    ax[0].bar_label(b, fmt="%.1f", fontsize=8, padding=2)
ax[0].set(xticks=xs, xticklabels=[n.split()[0] for n in GREFS],
          ylabel="TV, % of the national total",
          title="a  government category: every referent gives the same ordering")
ax[0].legend(fontsize=8, frameon=False)

# (b) New York under each referent and each allocator
lbl = [n.split()[0] for n in GREFS]
ax[1].bar(lbl, [GREFS[n]["NY"] * 100 for n in GREFS], color="#cccccc",
          edgecolor=CK, label="referent range")
for k in ["breadth", "gdp", "windc"]:
    ax[1].axhline(ALLOC[k]["G"]["NY"] * 100, color=COL[k], lw=1.8, ls="--",
                  label=LABEL[k])
ax[1].set(ylabel="New York share of government final demand, %",
          title="b  no referent puts New York near the WiNDC value")
ax[1].legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(FIG_DIR / "fd_gov_referents.png", dpi=160,
                                bbox_inches="tight")
plt.show()

---
## 6. Why the breadth share compresses

The bias is not an accident of the data: it follows from the definition. $\theta_s$ is the
**unweighted** mean of $S_{s,\cdot}$ over the 36 sectors, so a sector worth 30 bn\$ of output
carries the same weight as one worth 2 tn\$. The size-weighted mean of the same matrix is, by
construction, the state's share of national product. The breadth share therefore measures *how
broadly a state is present across sectors*, and states specialised in a few large sectors are
penalised exactly in proportion to that specialisation.

In [ ]:
S = S_matrix(YEAR)
w_j = (_sag[_sag.LineCode.isin(_L2S)].assign(sec=lambda d: d.LineCode.map(_L2S))
       .groupby("sec")[str(YEAR)].sum().reindex(SECS36))
w_j = w_j / w_j.sum()                                   # sector weight in national product
theta_w = S.mul(w_j, axis=1).sum(1); theta_w /= theta_w.sum()   # size-weighted mean = gdp share

fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.8))
ax[0].scatter(theta_w * 100, theta * 100, s=24, color=CO_, zorder=3)
lo, hi = theta_w.min() * 80, theta_w.max() * 120
ax[0].plot([lo, hi], [lo, hi], "k:", lw=.9)
for s in ["NY", "CA", "TX", "FL", "WY", "AK", "ND", "VT", "DC"]:
    ax[0].annotate(s, (theta_w[s] * 100, theta[s] * 100), textcoords="offset points",
                   xytext=(4, 3), fontsize=8)
sl = np.polyfit(np.log(theta_w), np.log(theta), 1)[0]
ax[0].set(xscale="log", yscale="log", xlabel="size-weighted sector mean of S  (= product share), %",
          ylabel=r"unweighted sector mean of $S$  ($\theta$), %",
          title=f"a  unweighted vs size-weighted mean of the same matrix  (ε={sl:.2f})")

contrib = (S.loc["NY"] - theta_w["NY"]) * (1 / len(SECS36))   # NY deviation, per sector
o = contrib.sort_values()
top = pd.concat([o.head(6), o.tail(6)])
cols = [CO_ if v < 0 else CB for v in top.values]
ax[1].barh([t[:34] for t in top.index], top.values * 100, color=cols)
ax[1].axvline(0, color="k", lw=.8)
gap = (theta["NY"] - theta_w["NY"]) * 100
ax[1].set(xlabel=r"contribution to $\theta_{NY}$ − product share, pp",
          title=f"b  largest sector contributions to New York's gap "
                f"(all {len(SECS36)} sum to {gap:+.2f} pp)")
ax[1].grid(axis="y", alpha=0)
fig.tight_layout(); fig.savefig(FIG_DIR / f"fd_breadth_mechanism_{YEAR}.png", dpi=160,
                                bbox_inches="tight")
plt.show()

print(f"NY: unweighted sector mean of S = {theta['NY']*100:.2f}%, "
      f"size-weighted = {theta_w['NY']*100:.2f}%")

# the mechanism, stated as a testable claim: the ratio theta/product share should track how
# evenly a state's own output is spread across sectors, not its size.
comp_s = S.mul(w_j, axis=1)                       # state x sector output, national units
comp_s = comp_s.div(comp_s.sum(1), axis=0)        # each state's own sector composition
even   = 1 - (comp_s ** 2).sum(1)                 # 1 - HHI: 0 = one sector, 1 = perfectly even
ratio  = np.log(theta / theta_w)                  # >0 : state inflated by the breadth share
print(f"corr( log(theta / product share), evenness of the state's own sector mix ) = "
      f"{np.corrcoef(ratio, even)[0,1]:+.3f}   (n=51)")
print(f"corr( log(theta / product share), state size )                            = "
      f"{np.corrcoef(ratio, np.log(theta_w))[0,1]:+.3f}")
RESULTS["breadth_ratio_vs_evenness_corr"] = float(np.corrcoef(ratio, even)[0, 1])
RESULTS["breadth_ratio_vs_size_corr"]     = float(np.corrcoef(ratio, np.log(theta_w))[0, 1])
RESULTS["theta_vs_weighted_elasticity"] = float(sl)

---
## 7. Stability over the delivered series, 1997–2022

A single year cannot support a change of method. The three allocators and the two referents are
recomputed for every year of the delivered series. WiNDC shares are taken from the state-level
(71-sector) files, which exist for all years; §1.1 measured the gap to their harmonised
counterparts, which is immaterial on the two categories that carry a referent.

In [ ]:
recs, ny_recs = [], []
for y in YEARS:
    try:
        w_y = windc_share(y, harmonised=False)
    except FileNotFoundError:
        continue
    al = {"breadth": pd.DataFrame({c: breadth_share(y) for c in WCATS}),
          "gdp":     pd.DataFrame({c: gdp_share(y)     for c in WCATS}),
          "windc":   w_y}
    rf = {"C": pce_share(y), "G": govva_share(y)}
    for cat, ref in rf.items():
        for k, A in al.items():
            recs.append(dict(year=y, category=cat, allocator=k, **metrics(A[cat], ref)))
    ny_recs.append(dict(year=y, referent=rf["C"]["NY"],
                        **{k: al[k]["C"]["NY"] for k in al}))
TS  = pd.DataFrame(recs)
NYS = pd.DataFrame(ny_recs).set_index("year")
print(f"years covered: {TS.year.min()}–{TS.year.max()}  ({TS.year.nunique()})")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for k in ALLOC:
    d = TS[(TS.category == "C") & (TS.allocator == k)]
    ax[0].plot(d.year, d.TV * 100, color=COL[k], lw=1.8, label=LABEL[k])
    d = TS[(TS.category == "G") & (TS.allocator == k)]
    ax[1].plot(d.year, d.TV * 100, color=COL[k], lw=1.8, label=LABEL[k])
    ax[2].plot(NYS.index, NYS[k] * 100, color=COL[k], lw=1.8, label=LABEL[k])
ax[2].plot(NYS.index, NYS.referent * 100, color=CK, lw=2.2, ls="--", label="referent (PCE)")
ax[0].set(xlabel="year", ylabel="TV, % of the national total",
          title="a  household category C, against BEA PCE")
ax[1].set(xlabel="year", ylabel="TV, % of the national total",
          title="b  government category G, against government value added")
ax[2].set(xlabel="year", ylabel="New York share, %",
          title="c  New York, household final demand")
for a in ax: a.legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(FIG_DIR / "fd_allocator_timeseries.png", dpi=160,
                                bbox_inches="tight")
plt.show()

summ = (TS.groupby(["category", "allocator"])[["TV", "beta", "eps", "rho"]]
          .agg(["mean", "min", "max"]))
display(summ.round(4))

gG = TS[(TS.category == "G") & (TS.allocator == "windc")].set_index("year").TV
print(f"WiNDC government shares drift away from the referent after 2000: TV "
      f"{gG.loc[1997:2000].mean()*100:.1f}% over 1997-2000, "
      f"{gG.loc[2010:2022].mean()*100:.1f}% over 2010-2022 — the only series of the six that is "
      f"not stationary. The two alternatives move by less than "
      f"{max(abs(TS[(TS.category=='G') & (TS.allocator==k)].set_index('year').TV.loc[2010:2022].mean() - TS[(TS.category=='G') & (TS.allocator==k)].set_index('year').TV.loc[1997:2000].mean()) for k in ['breadth','gdp'])*100:.1f} pp over the same span.")
RESULTS["windc_G_TV_9700_vs_1022"] = [float(gG.loc[1997:2000].mean()),
                                      float(gG.loc[2010:2022].mean())]
for _stat in ["TV", "beta", "eps"]:
    RESULTS[f"{_stat}_mean_1997_2022"] = {
        f"{c}/{a}": round(float(v), 5)
        for (c, a), v in TS.groupby(["category", "allocator"])[_stat].mean().items()}

---
## 8. Two refinements of candidate C

**(i) State $\times$ category $\times$ sector.** WiNDC also identifies the destination share of each
*good*, $\theta^{W}_{s,c,j}$. It is the most faithful allocator available and still conserves the
OECD totals exactly, provided the normalisation is taken within each $(c,j)$ pair. The obvious
objection is sparsity — thin national totals give unstable shares, and a state recorded as
absorbing nothing of a good would receive an exact zero. That objection is tested below and does
not hold: once the sector-category pairs that carry no mass at all are set aside, which are
structural (there is no investment in accommodation services), the exact zeros are a fraction of a
per cent of the state cells. The variant is well behaved.

What blocks it here is evidential, not numerical. Both referents of §1.3 are state totals summed
over goods, and re-aggregating $\theta^{W}_{s,c,j}$ over goods returns $\theta^{W}_{s,c}$ exactly
(checked below). The notebook therefore cannot discriminate between the two, and the finer variant
is recorded as a documented option rather than adopted on evidence this test does not provide.

**(ii) Hybrid.** §4 and §5 showed that the WiNDC government shares are the weak point. A hybrid **D** is
therefore evaluated, keeping WiNDC for $C$ and $I$ and allocating $G$ by the state product share,
one scalar per state.
Note the constraint on how such a hybrid may be scored: allocating $G$ by government value added,
which is the referent, would drive the measured government error to zero *by construction* and
would prove nothing. The hybrid is therefore built on the state product share, which is an
independent allocator already used elsewhere in the construction, so that its government score
remains a genuine out-of-sample comparison against the referent.

In [ ]:
sh_sec, sec_names = windc_share(YEAR, harmonised=True, by_sector=True)   # (sector, state, cat)
mass_sec = windc_F_dest(npz_h, by_sector=True)[0]                        # (sector, state, cat)
tot_sec  = mass_sec.sum(1)                                               # (sector, cat)

thin  = tot_sec < 1.0                                   # national total below 1 bn $
live  = tot_sec > 0                                     # pair carries any mass at all
zeros = (sh_sec == 0).sum((1,)) / len(STATES)           # fraction of states at exact zero
disp = pd.DataFrame({
    "national total bn$": tot_sec.reshape(-1),
    "states at zero %":   (zeros.reshape(-1) * 100),
}, index=pd.MultiIndex.from_product([sec_names, WCATS], names=["sector", "cat"]))
print(f"(c,j) pairs with no mass at all: {(~live).sum()} of {live.size} "
      f"({(~live).sum()/live.size*100:.0f}%) — structural, e.g. investment in accommodation")
print(f"(c,j) pairs with a national total below 1 bn$: {thin.sum()} of {thin.size}"
      f"  ({thin.sum()/thin.size*100:.0f}%), carrying "
      f"{tot_sec[thin].sum()/tot_sec.sum()*100:.2f}% of the mass")
w_cj = tot_sec / tot_sec.sum()
print(f"states at an exact zero, mean over the {live.sum()} (c,j) pairs that carry mass: "
      f"{zeros[live].mean()*100:.1f}%   (mass-weighted: {(zeros*w_cj)[live].sum()*100:.1f}%)")
display(disp.sort_values("national total bn$").head(6).round(2))
display(disp.sort_values("national total bn$").tail(4).round(2))

# mass-weighted state totals implied by the sector-specific variant, per category
imp_sec = np.einsum("jsc,jc->sc", sh_sec, tot_sec)
imp_sec = pd.DataFrame(imp_sec / imp_sec.sum(0), index=STATES, columns=WCATS)
print("\nsector-specific vs state-level WiNDC shares — max |difference| when re-aggregated:",
      f"{(imp_sec - windc).abs().values.max()*100:.3f} pp (identical by construction)")
RESULTS["thin_cj_pairs_pct_of_mass"]  = float(tot_sec[thin].sum() / tot_sec.sum() * 100)
RESULTS["mean_pct_states_at_zero"]    = float(zeros[live].mean() * 100)
RESULTS["massw_pct_states_at_zero"]   = float((zeros * w_cj)[live].sum() * 100)
RESULTS["empty_cj_pairs"]             = [int((~live).sum()), int(live.size)]

In [ ]:
ALLOC["hybrid"] = pd.DataFrame({"C": windc["C"], "I": windc["I"], "G": gdp_sh})
LABEL["hybrid"] = "D  retained (WiNDC C,I; state product share G)"; COL["hybrid"] = CP

# the circular variant, reported only to show why it is not used as evidence
ALLOC_circ = pd.DataFrame({"C": windc["C"], "I": windc["I"], "G": ref_G})
print("hybrid on government value added: TV against that same referent = "
      f"{metrics(ALLOC_circ['G'], ref_G)['TV']:.4f} — zero by construction, not evidence.")

rows = []
for cat, ref in REF.items():
    for k in ["breadth", "gdp", "windc", "hybrid"]:
        rows.append(dict(category=cat, allocator=k, **metrics(ALLOC[k][cat], ref, MASS[cat])))
MET4 = pd.DataFrame(rows).set_index(["category", "allocator"])

# mass-weighted composite over the two categories that have a referent
cover = MASS["C"] + MASS["G"]
comp = pd.Series({k: sum(MET4.loc[(c, k), "MIS_M"] for c in REF) / 1e3
                  for k in ["breadth", "gdp", "windc", "hybrid"]}).sort_values()
print(f"composite misallocation over C+G ({cover/1e6:,.1f} tn$ = "
      f"{cover/sum(MASS.values())*100:.0f}% of US final demand), bn$"
      f"  — the investment category ({MASS['I']/1e6:,.1f} tn$) carries no referent "
      f"and is excluded:")
display(comp.round(0).to_frame("misallocated bn$"))
display(MET4[["TV", "MIS_M", "beta", "eps", "rho"]]
        .style.format({"TV": "{:.4f}", "MIS_M": "{:,.0f}", "beta": "{:+.4f}",
                       "eps": "{:.3f}", "rho": "{:.4f}"}))
RESULTS["composite_misallocation_bn"] = comp.round(1).to_dict()

---
## 9. Effect on the delivered table

The allocator enters two blocks of the nested table: the imported final demand
$F^n_{(c,i),(s,\text{cat})}$ and the destination side of the domestic block
$F^n_{(s,i),(s',\text{cat})}$. The total final-demand *inflow* of a state is therefore
proportional to its allocator share within each category. The table below converts the share
errors into the quantity a user of the delivered files actually sees, for the retained
specification **D** (WiNDC shares for $C$ and $I$, production share for $G$) and, as a comparison,
for the all-WiNDC variant **C**.

In [ ]:
FD_tot = pd.Series(MASS)                                   # M$ per WiNDC category
def inflow(A):                                             # total FD received by each state, M$
    return (A[WCATS] * FD_tot).sum(1)

INF = pd.DataFrame({k: inflow(ALLOC[k]) for k in ["breadth", "gdp", "windc", "hybrid"]})
INF["delivered (A)"] = INF["breadth"]
delta = (INF[["windc", "hybrid", "gdp"]].sub(INF["breadth"], axis=0)) / 1e3   # bn$
rel   = INF[["windc", "hybrid", "gdp"]].div(INF["breadth"], axis=0) - 1

show = pd.DataFrame({
    "delivered A, bn$":  INF["breadth"] / 1e3,
    "retained D, bn$":   INF["hybrid"]  / 1e3,
    "Δ bn$":             delta["hybrid"],
    "Δ %":               rel["hybrid"] * 100,
    "C (all WiNDC) Δ %": rel["windc"] * 100,
}).sort_values("Δ bn$")
display(pd.concat([show.head(6), show.tail(8)]).round(1))

moved  = delta["hybrid"].abs().sum() / 2
movedC = delta["windc"].abs().sum() / 2
print(f"\nFinal-demand inflow reallocated, A -> D (retained): {moved:,.0f} bn$ "
      f"({moved*1e3/FD_tot.sum()*100:.1f}% of US final demand)")
print(f"                              A -> C (all WiNDC)  : {movedC:,.0f} bn$ "
      f"({movedC*1e3/FD_tot.sum()*100:.1f}%)")
for k, lab in [("hybrid", "D retained"), ("windc", "C all WiNDC")]:
    print(f"New York, {lab:<12}: {INF.loc['NY','breadth']/1e3:,.0f} -> "
          f"{INF.loc['NY',k]/1e3:,.0f} bn$   ({rel.loc['NY',k]*100:+.1f}%)")
RESULTS["delivered_reallocated_bn"] = {"A_to_D": float(moved), "A_to_C": float(movedC)}
RESULTS["NY_inflow_bn"] = {"A_breadth": float(INF.loc["NY", "breadth"] / 1e3),
                           "B_gdp":     float(INF.loc["NY", "gdp"] / 1e3),
                           "C_windc":   float(INF.loc["NY", "windc"] / 1e3),
                           "D_hybrid":  float(INF.loc["NY", "hybrid"] / 1e3)}

fig, ax = plt.subplots(figsize=(13, 4.4))
d = delta["hybrid"].sort_values()
ax.bar(d.index, d.values, color=[CO_ if v < 0 else CP for v in d.values], label="retained D")
ax.plot(d.index, delta["windc"].reindex(d.index).values, "o", ms=3.5, color=CG,
        label="all-WiNDC variant C")
ax.axhline(0, color="k", lw=.8); ax.legend(fontsize=8, frameon=False)
ax.set(ylabel="Δ final-demand inflow, bn $", title=
       f"Change in each state's final-demand inflow, breadth share → retained allocator, {YEAR}")
ax.tick_params(axis="x", labelsize=7); ax.grid(axis="x", alpha=0)
fig.tight_layout(); fig.savefig(FIG_DIR / f"fd_allocator_delivered_delta_{YEAR}.png",
                                dpi=160, bbox_inches="tight")
plt.show()

---
## 10. The arbitrage in one figure

Every candidate conserves the OECD national totals exactly, so the choice comes down to two
numbers: the mass misplaced on household consumption, measured against personal consumption
expenditure by state, and the mass misplaced on government consumption, measured against the four
benchmarks of §5. Plotting one against the other turns the arbitrage into a dominance question,
and the answer is unambiguous.

*Encoding note.* All four points share one plot, so identity would have to survive every pairwise
comparison at once. The four-hue house palette does not: the green and orange of the other figures
separate by only $\Delta E = 4.4$ under simulated deuteranopia, below the usable floor. Identity is
therefore carried by position and direct labels, and colour is spent on the one thing the figure
asserts — the retained allocator against the three that were rejected.

In [ ]:
ACCENT, INK, MUTED = "#8a5aa8", "#444444", "#8c8c88"   # one categorical hue + neutral ink

pts = {}
for k, lab in [("breadth", "A  breadth share $\\theta$"),
               ("gdp",     "B  state product share"),
               ("windc",   "C  sub-national shares,\n     all three categories"),
               ("hybrid",  "D  retained: sub-national for C and I,\n     state product for G")]:
    x  = MET4.loc[("C", k), "MIS_M"] / 1e3                      # household, single benchmark
    gk = "gdp" if k == "hybrid" else k                          # D uses the product share on G
    ys = np.array([GR.loc[n, f"TV {gk}"] / 100 * MASS["G"] / 1e3 for n in GREFS])
    pts[k] = dict(label=lab, x=x, y=ys.mean(), lo=ys.min(), hi=ys.max())

fig, ax = plt.subplots(figsize=(7.8, 5.8))
d = pts["hybrid"]

# the region every alternative would have to beat: worse than D on both axes
ax.add_patch(plt.Rectangle((d["x"], d["lo"]), 1e4, 1e4, facecolor=ACCENT, alpha=.07,
                           edgecolor="none", zorder=0))
ax.annotate("every point in this region is\ndominated by D: worse on both axes",
            (1400, 505), fontsize=8.5, color=MUTED, ha="right", va="top",
            style="italic", zorder=1)

for k, p in pts.items():
    hero = k == "hybrid"
    c = ACCENT if hero else INK
    ax.errorbar(p["x"], p["y"], yerr=[[p["y"] - p["lo"]], [p["hi"] - p["y"]]],
                fmt="none", ecolor=c, elinewidth=1.8 if hero else 1.1,
                capsize=4, alpha=1 if hero else .5, zorder=3)
    ax.plot(p["x"], p["y"], "o", ms=13 if hero else 8,
            mfc=ACCENT if hero else "white", mec=c, mew=2 if hero else 1.4, zorder=4)

for k, dx, dy, ha in [("breadth", -16, 26, "right"), ("gdp", 20, 16, "left"),
                      ("windc", 22, 2, "left")]:
    ax.annotate(pts[k]["label"], (pts[k]["x"], pts[k]["y"]), xytext=(dx, dy),
                textcoords="offset points", fontsize=9.5, color=INK, ha=ha)
ax.annotate(d["label"], (d["x"], d["lo"]), xytext=(22, -34), textcoords="offset points",
            fontsize=10, color=ACCENT, fontweight="bold")

ax.set(xlim=(0, 1430), ylim=(0, 530),
       xlabel="household consumption misplaced, bn $\n"
              "(against personal consumption expenditure by state)",
       ylabel="government consumption misplaced, bn $\n(range over the four benchmarks)")
ax.set_title("Every alternative is worse on at least one axis", fontsize=11.5,
             fontweight="bold", loc="left")
ax.grid(alpha=.22)
fig.tight_layout(); fig.savefig(FIG_DIR / f"fd_allocator_choice_{YEAR}.png", dpi=200,
                                bbox_inches="tight")
plt.show()

print(f"{'allocator':<9} {'household bn$':>14} {'government bn$, min-max':>26}")
for k, p in pts.items():
    print(f"  {k:<8} {p['x']:12,.0f}   {p['lo']:10,.0f} - {p['hi']:,.0f}")
print(f"\nD has the minimum on both axes, so no other candidate dominates it.")
print(f"Against the incumbent A: {pts['breadth']['x']/d['x']:.0f}x less household mass misplaced "
      f"and {pts['breadth']['y']/d['y']:.1f}x less government mass.")
print(f"Against C, which differs from D only on the government category: same household figure, "
      f"{pts['windc']['y']/d['y']:.1f}x less government mass.")
RESULTS["choice_frontier_bn"] = {k: {"C": round(p["x"], 1), "G_lo": round(p["lo"], 1),
                                     "G_hi": round(p["hi"], 1)} for k, p in pts.items()}

---
### 10.1 Trade-offs, allocator by allocator

In [ ]:
trade = pd.DataFrame([
 ("A  breadth share θ",
  "state only",
  "single public source (SAGDP2) for all sectors, states and years; exact conservation",
  "not a measure of final demand at all; unweighted sector mean compresses the "
  "distribution towards 1/51; no category detail",
  "large consuming states under-allocated, small diversified states over-allocated"),
 ("B  state product share",
  "state only",
  "same source and coverage as A; a genuine size measure, no compression",
  "measures where output is produced, not where it is absorbed; no category detail",
  "over-allocates production-heavy and headquarters states"),
 ("C  WiNDC final-demand shares",
  "state × category; state × category × sector available",
  "the only candidate that measures the concept being allocated; carries the official "
  "regional statistic per category (PCE for C, SGF for G, GSP for I); category detail",
  "3 categories only, the 6-way OECD split is not identified; government shares inherit the "
  "State Government Finances key, which does not describe federal or local purchases",
  "government category misplaced (see §3); C and I sound"),
 ("D  retained: C for C and I, state product share for G",
  "state × category",
  "keeps the gain of C where it is measured and removes its documented weak point; adds no "
  "input the construction does not already use",
  "mixes two provenances in one block; the G allocator measures where output is produced, "
  "not where government buys",
  "residual error on G: 7.7% of the category, against 16.8% for C and 12.1% for A"),
], columns=["allocator", "granularity", "in favour", "against", "residual bias"])
pd.set_option("display.max_colwidth", 58)
display(trade)
trade.to_csv(FIG_DIR / "fd_allocator_tradeoffs.csv", index=False)

(FIG_DIR / f"fd_allocator_results_{YEAR}.json").write_text(json.dumps(RESULTS, indent=1,
                                                                     default=float))
print("results ->", (FIG_DIR / f'fd_allocator_results_{YEAR}.json').relative_to(ROOT))

---
## 11. Conclusion

The paragraph below is generated from the measured values, so that no number reaching the
manuscript can drift away from the notebook that produced it. It is written for the Technical
Validation, with the companion replacement for the corresponding Usage Note.